# 03 — Joins, Broadcast, and Skew

Join semantics, the physical join strategies Spark picks under the hood, and data skew — arguably the single most-asked performance topic in data engineer interviews.

> **Setup note:** these notebooks are written but **not executed** — PySpark is not
> installed in this environment. To run them locally:
>
> ```bash
> python -m venv .venv && source .venv/bin/activate
> pip install pyspark==3.5.1
> # Java 11/17 must be on PATH (java -version)
> jupyter notebook
> ```
>
> Everything below is correct, runnable PySpark — read it as a reference and run
> cell-by-cell once your environment is set up.

In [ ]:
employees = spark.createDataFrame(
    [(1, "alice", 10), (2, "bob", 10), (3, "cara", 20), (4, "dan", None)],
    ["emp_id", "name", "dept_id"],
)
departments = spark.createDataFrame(
    [(10, "engineering"), (20, "sales"), (30, "marketing")],
    ["dept_id", "dept_name"],
)

## 1. Join types (logical semantics)

- **inner** — rows with matching keys on both sides only.
- **left / left_outer** — all left rows; unmatched right columns become `null`.
- **right / right_outer** — mirror of left.
- **full / outer** — union of left and right, nulls on the unmatched side.
- **left_semi** — like inner, but returns **only left columns**, and at most one copy of each left row even if there are multiple right matches. Equivalent to a SQL `WHERE EXISTS`.
- **left_anti** — left rows with **no** match on the right. Equivalent to `WHERE NOT EXISTS`.
- **cross** — Cartesian product; every left row paired with every right row. Dangerous at scale — Spark requires an explicit `.crossJoin()` or a config flag to avoid accidental Cartesian products from a missing join key.

In [ ]:
employees.join(departments, "dept_id", "inner").show()
employees.join(departments, "dept_id", "left").show()
employees.join(departments, "dept_id", "left_semi").show()   # only employee columns
employees.join(departments, "dept_id", "left_anti").show()   # employees with no dept match (dan)

## 2. Physical join strategies — what actually runs on the cluster

Spark's planner picks one of these based on data size, join keys, and config, regardless of which logical join type you wrote:

- **Broadcast Hash Join (BHJ)** — the smaller side is sent whole to every executor (no shuffle of the large side). Fastest option, used automatically when the small side is below `spark.sql.autoBroadcastJoinThreshold` (default 10 MB). You can force it with `broadcast()`.
- **Sort-Merge Join (SMJ)** — both sides are shuffled and sorted by the join key, then merged. The default for large-large joins; requires a full shuffle of both sides — expensive but scales to any size.
- **Shuffle Hash Join (SHJ)** — both sides shuffled, then a hash table is built on one side per partition. Used less often than SMJ in modern Spark (SMJ is generally preferred/more robust); only chosen when one side is small-ish but over the broadcast threshold and hashing is cheaper than sorting.

**Interview answer:** "Spark broadcasts the small side when it fits in memory to avoid shuffling the big side entirely; otherwise it falls back to a sort-merge join that shuffles both sides by the join key."

In [ ]:
from pyspark.sql.functions import broadcast

# Force a broadcast join explicitly (good practice when you *know* one side is small,
# rather than relying on the size-based heuristic / stats being accurate)
plan = employees.join(broadcast(departments), "dept_id", "inner")
plan.explain(mode="formatted")   # look for "BroadcastHashJoin" in the physical plan

print(spark.conf.get("spark.sql.autoBroadcastJoinThreshold"))  # default: 10485760 (10MB)

## 3. Data skew — what it is and how to fix it

**Symptom:** in the Spark UI, a stage has hundreds of tasks that finish in seconds and one or two "straggler" tasks that take 100x longer — because one join key (e.g. `user_id = NULL`, or a dominant customer) has wildly more rows than the others, and all of them hash to the same partition/reducer.

**Fixes:**
1. **Broadcast the small side** if one side of the skewed join is small enough — sidesteps the shuffle entirely.
2. **Salting** — add a random suffix to the skewed key on both sides to spread the hot key across many partitions, join, then strip the salt.
3. **Adaptive Query Execution (AQE) skew join optimization** (`spark.sql.adaptive.skewJoin.enabled=true`, on by default in Spark 3.x) — Spark detects skewed partitions at runtime from shuffle statistics and automatically splits them into smaller sub-partitions.
4. **Isolate and handle the hot key separately** — filter it out, broadcast-join just that key's rows, union back with the sort-merge-joined rest.

In [ ]:
from pyspark.sql.functions import concat_ws, floor, rand, explode, array, lit as spark_lit

# Manual salting pattern (useful even where AQE isn't available, e.g. older clusters)
N_SALT = 8

left_salted = employees.withColumn(
    "salt", floor(rand() * N_SALT).cast("int")
).withColumn("dept_id_salted", concat_ws("_", col("dept_id"), col("salt")))

# explode the small/skewed side across every salt bucket so every salted key on
# the left side finds its match somewhere on the right
right_salted = departments.withColumn(
    "salt", explode(array([spark_lit(i) for i in range(N_SALT)]))
).withColumn("dept_id_salted", concat_ws("_", col("dept_id"), col("salt")))

salted_join = left_salted.join(right_salted, "dept_id_salted").drop("salt", "dept_id_salted")
salted_join.show()

## 4. Interview Q&A

1. **"A join is taking forever and the Spark UI shows one huge task — what's going on and how do you fix it?"** — data skew on the join key; broadcast the small side, salt the key, or rely on/verify AQE skew handling.
2. **"When would Spark *not* use a broadcast join even though one side is small?"** — if size statistics are stale/unavailable (e.g. reading from a source without stats, or after several transformations Spark can't size accurately) — force it explicitly with `broadcast()`.
3. **"Why avoid `left_anti`/`left_semi` reimplemented as `filter(isin(...))` with a collected list?"** — collecting a large key list to the driver and using `.isin()` doesn't scale and moves work off the cluster; a proper semi/anti join stays distributed.
4. **"What's a Cartesian product and why is Spark cautious about it?"** — a cross join where every row pairs with every row (`|L| x |R|` output); usually the accidental result of a join condition that isn't actually selective (or a missing `on` clause) — Spark requires `spark.sql.crossJoin.enabled` or an explicit `.crossJoin()` to guard against it.

## Summary

- Logical join type (inner/left/semi/anti) is about *which rows* come out; physical strategy (broadcast/sort-merge/shuffle-hash) is about *how* Spark computes it.
- Broadcast the small side whenever possible — it avoids shuffling the large side entirely.
- Skew shows up as one straggler task; fix with broadcasting, salting, or AQE.
- Next: `04_aggregations_and_window_functions.ipynb`.